# 00 环境检查

运行本 notebook。若最后没有报错，说明本地 CPU 环境可用。

In [1]:
# 导入依赖并准备通用通信仿真工具函数。
from pathlib import Path
import sys

# 从当前目录向上查找 src/sionna，优先使用本仓库源码。
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "src" / "sionna").exists():
        sys.path.insert(0, str(p / "src"))
        break

import numpy as np
import torch
import matplotlib.pyplot as plt
from sionna.phy.mapping import Constellation, Mapper, Demapper
from sionna.phy.channel import AWGN, RayleighBlockFading

# 固定使用 CPU，保证没有 GPU 的学生电脑也能运行。
DEVICE = "cpu"
PRECISION = "single"

# 固定随机种子，便于复现实验结果。
torch.manual_seed(7)
np.random.seed(7)


# 将 Eb/N0 转成 AWGN 模块需要的噪声功率谱密度 no。
def ebno_db_to_no(ebno_db, nbits, coderate=1.0):
    ebno = 10 ** (torch.as_tensor(ebno_db, dtype=torch.float32, device=DEVICE) / 10.0)
    return 1.0 / (ebno * nbits * coderate)


# Demapper 输出 LLR，这里统一转换成 0/1 硬判决。
def hard_bits(llr):
    return (llr > 0).to(torch.int32)


# 统计误比特数，BER 会在此基础上除以总 bit 数。
def count_bit_errors(b, b_hat):
    return int(torch.sum(b.to(torch.int32) != b_hat.to(torch.int32)).item())


def ber(b, b_hat):
    return count_bit_errors(b, b_hat) / b.numel()


# 根据每符号比特数创建星座、映射器和解映射器。
def get_modem(nbits):
    if nbits == 1:
        points = torch.tensor([-1 + 0j, 1 + 0j], dtype=torch.complex64, device=DEVICE)
        const = Constellation(
            "custom", 1, points=points, precision=PRECISION, device=DEVICE
        )
    else:
        const = Constellation("qam", nbits, precision=PRECISION, device=DEVICE)
    return (
        const,
        Mapper(constellation=const, precision=PRECISION, device=DEVICE),
        Demapper(
            "app",
            constellation=const,
            hard_out=False,
            precision=PRECISION,
            device=DEVICE,
        ),
    )

In [2]:
# 打印版本并跑一条最小 QPSK 链路，确认环境和基本算子可用。
import sionna

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Sionna:", sionna.__version__)
print("Device:", DEVICE)

nbits = 2
const, mapper, demapper = get_modem(nbits)
b = torch.randint(0, 2, (128, 512), dtype=torch.int32, device=DEVICE)
no = ebno_db_to_no(8.0, nbits)
y = AWGN(precision=PRECISION, device=DEVICE)(mapper(b), no)
b_hat = hard_bits(demapper(y, no))

print("BER:", ber(b, b_hat))

assert DEVICE == "cpu"

assert ber(b, b_hat) < 0.10

Python: 3.12.4
PyTorch: 2.11.0+cpu
Sionna: 2.2.0
Device: cpu
BER: 0.00030517578125
